# Chapter 12: five checks and a quarantine table

Section 12.4 runs Kestrel's five checks on the synthetic slice and writes a quarantine table you can inspect.
The rules live in kestrel.run_checks: completeness, range, join, freshness and balance. A red tile means
"do not trust this number yet". It never means hold or release a batch.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import kestrel as k

con = k.connect()
quarantine = k.run_checks(con)
print(quarantine.to_string(index=False))

The data quality page: one tile per check, green or red, with the count behind it.

In [ ]:
for check, owner in k.OWNERS.items():
    n = (quarantine.check_failed == check).sum()
    print(f"{check:13s} {'RED' if n else 'green':5s} {n:2d} rows   owner: {owner}")

Write the table. Fix each row at the source: the LIMS, the tag map or the transfer log. Track the age of the
oldest row.

In [ ]:
k.OUT.mkdir(exist_ok=True)
quarantine.to_csv(k.OUT / "quarantine.csv", index=False)
oldest = pd.to_datetime(quarantine.date).min()
print(f"wrote out/quarantine.csv; oldest row dated {oldest:%d %B %Y}")